# 01 — Preprocessing and SGI calculation

This notebook reproduces the revised Standardized Groundwater level Index (SGI) workflow used in the manuscript.

**Procedure**

- trailing 3-, 6-, and 12-month arithmetic means;
- only complete consecutive calendar-month windows;
- non-parametric normal-score transformation separately for each calendar month;
- plotting position: \(p=(rank-0.5)/n\);
- \(SGI=\Phi^{-1}(p)\);
- average ranks for ties;
- group SGI calculated as the mean of available well-level SGI values;
- a group-level value is retained only when at least 50% of group wells and at least two wells are available.

The notebook exports:

- `data_processed/SGI_normal_score_long.csv`
- `data_processed/group_SGI.csv`


In [1]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
from scipy.stats import rankdata, norm

ROOT = Path.cwd()
if not (ROOT / "data_input").exists() and (ROOT.parent / "data_input").exists():
    ROOT = ROOT.parent

INPUT_DIR = ROOT / "data_input"
OUTPUT_DIR = ROOT / "data_processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GW_FILE = INPUT_DIR / "groundwater_levels_monthly.csv"
META_FILE = INPUT_DIR / "metadata_piezometers.csv"

print(f"Repository root: {ROOT.resolve()}")
print(f"Input directory: {INPUT_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Repository root: /mnt/data/repo_validation
Input directory: /mnt/data/repo_validation/data_input
Output directory: /mnt/data/repo_validation/data_processed


In [2]:
gw = pd.read_csv(GW_FILE, parse_dates=["date"], float_precision="round_trip")
metadata = pd.read_csv(META_FILE, float_precision="round_trip")

required_gw = {"date", "piezometer_id", "groundwater_level_m", "cluster_id"}
required_meta = {"piezometer_id", "cluster_id", "aquifer_group"}

missing_gw = required_gw.difference(gw.columns)
missing_meta = required_meta.difference(metadata.columns)
if missing_gw:
    raise ValueError(f"Missing groundwater columns: {sorted(missing_gw)}")
if missing_meta:
    raise ValueError(f"Missing metadata columns: {sorted(missing_meta)}")

gw = (
    gw.loc[:, ["date", "piezometer_id", "groundwater_level_m", "cluster_id"]]
      .dropna(subset=["date", "piezometer_id", "groundwater_level_m", "cluster_id"])
      .sort_values(["piezometer_id", "date"])
      .reset_index(drop=True)
)

if gw.duplicated(["piezometer_id", "date"], keep=False).any():
    raise ValueError("Duplicate well-month observations found in groundwater input.")

print(f"Groundwater observations: {len(gw):,}")
print(f"Piezometers: {gw['piezometer_id'].nunique()}")
print(metadata.groupby(["cluster_id", "aquifer_group"]).size())

Groundwater observations: 4,082
Piezometers: 16
cluster_id  aquifer_group        
1           shallow_unconfined       10
2           intermediate_confined     4
3           deep_confined             2
dtype: int64


In [3]:
def complete_trailing_mean(group: pd.DataFrame, window_months: int) -> pd.DataFrame:
    """Calculate arithmetic means for complete consecutive calendar-month windows."""
    monthly = (
        group.set_index("date")["groundwater_level_m"]
             .sort_index()
    )
    value_by_date = monthly.to_dict()
    records = []

    for end_date in monthly.index:
        expected_dates = pd.date_range(
            end=end_date,
            periods=window_months,
            freq="MS",
        )
        if all(date in value_by_date for date in expected_dates):
            values = [value_by_date[date] for date in expected_dates]
            records.append(
                {
                    "date": end_date,
                    "rolling_groundwater_level": float(np.mean(values)),
                }
            )

    return pd.DataFrame(records)


def normal_score(values: pd.Series) -> np.ndarray:
    x = values.to_numpy(dtype=float)
    ranks = rankdata(x, method="average")
    p = (ranks - 0.5) / len(x)
    return norm.ppf(p)


sgi_parts = []

for piezometer_id, well_data in gw.groupby("piezometer_id", sort=True):
    cluster_values = well_data["cluster_id"].astype(str).unique()
    if len(cluster_values) != 1:
        raise ValueError(f"More than one cluster_id assigned to {piezometer_id}: {cluster_values}")
    cluster_id = cluster_values[0]

    for window in (3, 6, 12):
        rolling = complete_trailing_mean(well_data, window)
        rolling["calendar_month"] = rolling["date"].dt.month
        rolling["SGI_normal_score"] = (
            rolling.groupby("calendar_month")["rolling_groundwater_level"]
                   .transform(normal_score)
        )

        out = rolling.drop(columns=["calendar_month"])
        out.insert(0, "window_months", window)
        out.insert(0, "cluster_id", cluster_id)
        out.insert(0, "piezometer_id", piezometer_id)
        sgi_parts.append(out)

sgi = pd.concat(sgi_parts, ignore_index=True)
sgi = sgi[
    [
        "piezometer_id",
        "cluster_id",
        "window_months",
        "date",
        "rolling_groundwater_level",
        "SGI_normal_score",
    ]
].sort_values(["piezometer_id", "window_months", "date"]).reset_index(drop=True)

sgi_path = OUTPUT_DIR / "SGI_normal_score_long.csv"
sgi.to_csv(sgi_path, index=False)
print(f"Saved {len(sgi):,} well-level SGI records to {sgi_path}")

Saved 11,653 well-level SGI records to /mnt/data/repo_validation/data_processed/SGI_normal_score_long.csv


In [4]:
group_sizes = (
    metadata.assign(cluster_id=metadata["cluster_id"].astype(str))
            .groupby("cluster_id")["piezometer_id"]
            .nunique()
            .to_dict()
)

group_sgi = (
    sgi.assign(cluster_id=sgi["cluster_id"].astype(str))
       .groupby(["cluster_id", "window_months", "date"], as_index=False)
       .agg(
           available_wells=("piezometer_id", "nunique"),
           group_mean_SGI_raw=("SGI_normal_score", "mean"),
       )
       .rename(columns={"cluster_id": "group"})
)

group_sgi["total_group_wells"] = group_sgi["group"].map(group_sizes).astype(int)
group_sgi["minimum_required_wells"] = group_sgi["total_group_wells"].apply(
    lambda n: max(2, math.ceil(0.5 * n))
)
group_sgi["group_mean_SGI"] = group_sgi["group_mean_SGI_raw"].where(
    group_sgi["available_wells"] >= group_sgi["minimum_required_wells"]
)

group_sgi = (
    group_sgi[
        [
            "group",
            "window_months",
            "date",
            "available_wells",
            "total_group_wells",
            "group_mean_SGI",
        ]
    ]
    .sort_values(["group", "window_months", "date"])
    .reset_index(drop=True)
)

group_path = OUTPUT_DIR / "group_SGI.csv"
group_sgi.to_csv(group_path, index=False)

print(f"Saved {len(group_sgi):,} group-level SGI records to {group_path}")
for group, total in sorted(group_sizes.items(), key=lambda x: int(x[0])):
    print(f"Group {group}: minimum {max(2, math.ceil(0.5 * total))} of {total} wells")

Saved 3,251 group-level SGI records to /mnt/data/repo_validation/data_processed/group_SGI.csv
Group 1: minimum 5 of 10 wells
Group 2: minimum 2 of 4 wells
Group 3: minimum 2 of 2 wells


In [5]:
qc = (
    sgi.groupby(["cluster_id", "window_months"])
       .agg(
           records=("SGI_normal_score", "size"),
           first_date=("date", "min"),
           last_date=("date", "max"),
           minimum_SGI=("SGI_normal_score", "min"),
           maximum_SGI=("SGI_normal_score", "max"),
       )
)
display(qc)

retained = (
    group_sgi.assign(retained=group_sgi["group_mean_SGI"].notna())
             .groupby(["group", "window_months"])["retained"]
             .agg(["sum", "count"])
)
display(retained)

records first_date  ... minimum_SGI  maximum_SGI
cluster_id window_months                      ...                         
1          3                 1942 2004-09-01  ...   -1.959964     1.959964
           6                 1893 2004-12-01  ...   -1.959964     1.959964
           12                1804 2005-06-01  ...   -1.914506     1.914506
2          3                 1324 1985-09-01  ...   -2.231606     2.231606
           6                 1291 1985-12-01  ...   -2.231606     2.231606
           12                1225 1986-06-01  ...   -2.200411     2.200411
3          3                  748 1990-09-01  ...   -2.177923     2.177923
           6                  730 1990-12-01  ...   -2.177923     2.177923
           12                 696 1991-06-01  ...   -2.153875     2.153875

[9 rows x 5 columns]

sum  count
group window_months            
1     3              224    236
      6              220    233
      12             208    227
2     3              433    464
      6              412    461
      12             373    451
3     3              347    401
      6              335    395
      12             313    383